# Chapter 3 — Customer Behaviour
Queries `mart_customer_behaviour` from BigQuery and exports Plotly chart JSON for the webpage.

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from google.cloud import bigquery
from google.oauth2 import service_account

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
load_dotenv(os.path.join(project_root, '.env'))

project_id  = os.getenv('GCP_PROJECT_ID')
creds_path  = os.path.join(project_root, os.getenv('GOOGLE_APPLICATION_CREDENTIALS'))
credentials = service_account.Credentials.from_service_account_file(creds_path)
client      = bigquery.Client(credentials=credentials, project=project_id)

OUT = os.path.join(project_root, 'outputs')
os.makedirs(OUT, exist_ok=True)
print('Connected to BigQuery ✓')

In [ ]:
df = client.query(f"""
    SELECT * FROM `{project_id}.olist_raw.mart_customer_behaviour`
    ORDER BY total_orders DESC
""").to_dataframe()
df.head()

In [ ]:
# Chart 1 — Orders by State
by_state = df.groupby('customer_state').agg(
    total_orders=('total_orders', 'sum'),
    total_customers=('total_customers', 'sum'),
    total_revenue=('total_revenue', 'sum'),
    avg_order_value=('avg_order_value', 'mean'),
    avg_review_score=('avg_review_score', 'mean')
).reset_index().sort_values('total_orders', ascending=False)

fig1 = px.bar(
    by_state, x='customer_state', y='total_orders',
    title='Total Orders by Customer State',
    labels={'customer_state': 'State', 'total_orders': 'Total Orders'},
    color='total_orders',
    color_continuous_scale='Blues'
)
fig1.update_layout(template='plotly_white', coloraxis_showscale=False)
fig1.show()
with open(os.path.join(OUT, 'customer_orders_by_state.json'), 'w') as f:
    f.write(fig1.to_json())
print('Exported customer_orders_by_state.json')

In [ ]:
# Chart 2 — Top 15 Cities by Orders
top_cities = df.groupby('customer_city').agg(
    total_orders=('total_orders', 'sum'),
    total_customers=('total_customers', 'sum')
).reset_index().sort_values('total_orders', ascending=False).head(15)

fig2 = px.bar(
    top_cities, x='total_orders', y='customer_city', orientation='h',
    title='Top 15 Cities by Order Volume',
    labels={'total_orders': 'Total Orders', 'customer_city': 'City'},
    color='total_orders',
    color_continuous_scale='Teal'
)
fig2.update_layout(template='plotly_white', coloraxis_showscale=False, yaxis={'categoryorder': 'total ascending'})
fig2.show()
with open(os.path.join(OUT, 'customer_top_cities.json'), 'w') as f:
    f.write(fig2.to_json())
print('Exported customer_top_cities.json')

In [ ]:
# Chart 3 — Avg Review Score by State
fig3 = px.bar(
    by_state.sort_values('avg_review_score', ascending=True),
    x='avg_review_score', y='customer_state', orientation='h',
    title='Average Review Score by Customer State',
    labels={'avg_review_score': 'Avg Review Score', 'customer_state': 'State'},
    color='avg_review_score',
    color_continuous_scale='RdYlGn',
    range_color=[3.5, 5]
)
fig3.update_layout(template='plotly_white', coloraxis_showscale=False)
fig3.show()
with open(os.path.join(OUT, 'customer_review_by_state.json'), 'w') as f:
    f.write(fig3.to_json())
print('Exported customer_review_by_state.json')

In [ ]:
# Chart 4 — Revenue vs Orders bubble by State
fig4 = px.scatter(
    by_state, x='total_orders', y='total_revenue',
    size='total_customers', text='customer_state',
    title='Revenue vs Orders by State',
    labels={'total_orders': 'Total Orders', 'total_revenue': 'Total Revenue (BRL)', 'total_customers': 'Customers'},
    color='avg_order_value',
    color_continuous_scale='Viridis'
)
fig4.update_traces(textposition='top center')
fig4.update_layout(template='plotly_white')
fig4.show()
with open(os.path.join(OUT, 'customer_revenue_vs_orders.json'), 'w') as f:
    f.write(fig4.to_json())
print('Exported customer_revenue_vs_orders.json')